# Metadata shortcuts and operating points
Attach the frozen extension inputs and the MSLD v2.0 image release for feature verification. Run on CPU.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, time, zipfile
STARTED = time.time()
EXPECTED = '72ca98e5bd5f5b6d1e64ca56786bb6513cdda952c21057d89bac948f7fd27fa6'
KAGGLE = Path('/kaggle/input').is_dir()
SEARCH = Path('/kaggle/input') if KAGGLE else Path.cwd()
WORK = Path('/kaggle/working') if KAGGLE else Path.cwd()
WORK.mkdir(parents=True,exist_ok=True)
def digest(p):
    with p.open('rb') as s:
        h=hashlib.sha256()
        for b in iter(lambda:s.read(2**20),b''):h.update(b)
        return h.hexdigest()
def verify(folder):
    manifest=folder/'extension_manifest.json'
    if not manifest.is_file() or digest(manifest)!=EXPECTED:return False
    for name,h in json.loads(manifest.read_text()).items():
        file=(folder/name).resolve()
        if not file.is_relative_to(folder.resolve()) or not file.is_file() or digest(file)!=h:
            raise ValueError('Missing or changed frozen file: '+name)
    return True
def locate():
    for p in SEARCH.rglob('extension_manifest.json'):
        if verify(p.parent):return p.parent
    for pattern in ('MSLD_Extension_Inputs.zip','MSLD_Extension_Inputs.bin'):
        for archive in SEARCH.rglob(pattern):
            target=Path(tempfile.mkdtemp(prefix='msld_extension_'))
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    dest=(target/member.filename).resolve()
                    if not dest.is_relative_to(target.resolve()):raise ValueError('Invalid archive path')
                    if member.is_dir():continue
                    dest.parent.mkdir(parents=True,exist_ok=True)
                    with z.open(member) as src,dest.open('wb') as dst:shutil.copyfileobj(src,dst)
            if verify(target):return target
    raise FileNotFoundError('Attach MSLD_Extension_Inputs.zip or place it beside the notebook.')
PACKAGE=locate()
print('Frozen inputs verified.',EXPECTED)

def restore(owner):
    destination=WORK/('EXT_'+owner)
    prior=[]
    for p in SEARCH.rglob('input_identity.json'):
        record=json.loads(p.read_text())
        if p.parent.resolve()!=destination.resolve() and record.get('worker')==destination.name and record.get('input_manifest_sha256')==EXPECTED:
            prior.append(p.parent)
    if len(prior)>1:raise ValueError('Attach only this worker’s latest full output version.')
    if prior and not destination.exists():
        index=prior[0]/'result_index.json'
        if index.is_file():
            for rel,h in json.loads(index.read_text())['files'].items():
                source=(prior[0]/rel).resolve()
                if not source.is_relative_to(prior[0].resolve()) or not source.is_file() or digest(source)!=h:
                    raise ValueError('Incomplete or changed previous output: '+rel)
        shutil.copytree(prior[0],destination)
    destination.mkdir(exist_ok=True)
    return destination



In [ ]:
import importlib.util
needed={'numpy':'numpy','pandas':'pandas','scipy':'scipy','sklearn':'scikit-learn','PIL':'Pillow','joblib':'joblib','threadpoolctl':'threadpoolctl'}
missing=[pkg for mod,pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)


In [ ]:
subprocess.run([sys.executable,str(PACKAGE/'code/verify_features.py'),str(PACKAGE),str(SEARCH)],check=True)

In [ ]:
DEST=restore('CPU')
subprocess.run([sys.executable,str(PACKAGE/'code/cpu_study.py'),str(PACKAGE),str(DEST),str(STARTED+9.5*3600 if KAGGLE else float('inf'))],check=True)
print('CPU COMPLETE:',json.loads((DEST/'result_index.json').read_text())['complete'])